# Model selection: Ettin + Decide heads on the realistic data

Ettin-150M (seeds 1-3) and Ettin-400M (seed 1) with the Decide heads, trained on the frozen train split, model selection on the validation split only. Exports GGUF Q8_0 / Q4_0 + heads.

In [ ]:
# Setup: the experiment branch (public), dependencies, the frozen split
import os, subprocess, hashlib, json
if not os.path.exists("/tmp/codeply"):
    subprocess.run(["git", "clone", "--depth", "1", "-b", "codeply-li-dtw-experiment",
                    "https://github.com/AwaisSDev/Codeply-Craft", "/tmp/codeply"], check=True)
os.chdir("/tmp/codeply/research/dtw")  # outside /kaggle/working: the output keeps only artifacts
subprocess.run("pip -q install 'transformers>=4.48' peft accelerate sentencepiece pyyaml", shell=True, check=True)
subprocess.run("cd data/real && for f in *.gz; do gunzip -kf $f; done", shell=True, check=True)
split = json.load(open("data/real/SPLIT.json"))
assert split["leakChecksPass"], "leak checks failed: do not train"
for k in ("train", "val", "test"):
    assert hashlib.sha256(open(f"data/real/{k}.jsonl", "rb").read()).hexdigest() == split["hashes"][k], f"{k} changed"
print({k: v["n"] for k, v in split["splits"].items()})
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# llama.cpp at the pinned commit: GGUF converter and quantizer
import os, subprocess
if not os.path.exists("/tmp/llama.cpp"):
    subprocess.run("git clone https://github.com/ggml-org/llama.cpp /tmp/llama.cpp && cd /tmp/llama.cpp && git checkout aa588a3463357ee990b314f41dda242869130704", shell=True, check=True)
    subprocess.run("cd /tmp/llama.cpp && cmake -B build -DCMAKE_BUILD_TYPE=Release -DLLAMA_CURL=OFF -DLLAMA_BUILD_TESTS=OFF -DLLAMA_BUILD_EXAMPLES=OFF > /dev/null && cmake --build build -j4 --target llama-quantize > /dev/null", shell=True, check=True)
    subprocess.run("pip -q install -e /tmp/llama.cpp/gguf-py", shell=True, check=True)
def export_gguf(run):
    subprocess.run(f"python /tmp/llama.cpp/convert_hf_to_gguf.py {run}/backbone --outtype f16 --outfile {run}/backbone-f16.gguf", shell=True, check=True)
    for q in ("Q8_0", "Q4_0"):
        subprocess.run(f"/tmp/llama.cpp/build/bin/llama-quantize {run}/backbone-f16.gguf {run}/backbone-{q.lower()}.gguf {q}", shell=True, check=True)
    subprocess.run(f"python model/export_heads.py {run}", shell=True, check=True)

In [ ]:
import subprocess
import shutil
# (name, model, seed, micro-batch, accumulation): effective batch 32 for every run. Ettin-400M needs micro-batch 2 on a
# T4 (micro-batch 8 ran out of memory in run 3, after the three 150M seeds had finished).
RUNS = [("ettin150", "jhu-clsp/ettin-encoder-150m", 1, 8, 4), ("ettin150", "jhu-clsp/ettin-encoder-150m", 2, 8, 4),
        ("ettin150", "jhu-clsp/ettin-encoder-150m", 3, 8, 4), ("ettin400", "jhu-clsp/ettin-encoder-400m", 1, 2, 16)]
ONLY = os.environ.get("ONLY_RUNS", "")  # e.g. "real-ettin400-s1" to rerun only that one
for name, hf, seed, bs, accum in RUNS:
    run = f"runs/real-{name}-s{seed}"
    if os.path.exists(f"{run}/heads.json") or (ONLY and run.split("/")[1] not in ONLY.split(",")):
        continue
    # Effective batch 32 as pre-registered: 8 x 4 accumulation + gradient checkpointing. The first run (bs 32)
    # ran out of memory on the T4 (16 GB): realistic inputs reach 1,462 tokens and T4 has no FlashAttention.
    subprocess.run(f"cd model && PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python train_encoder.py --model {hf} "
                   f"--train ../data/real/train.spans.jsonl --val ../data/real/val.spans.jsonl "
                   f"--out ../{run} --epochs 3 --bs {bs} --accum {accum} --grad-ckpt --seed {seed}", shell=True, check=True)
    export_gguf(run)
    # Save this run's artifacts at once, so a later failure cannot lose them.
    out = f"/kaggle/working/runs/{run.split('/')[1]}"
    os.makedirs(out, exist_ok=True)
    for f in ("heads.bin", "heads.json", "config.json", "train-log.jsonl", "backbone-q8_0.gguf", "backbone-q4_0.gguf"):
        if os.path.exists(f"{run}/{f}"):
            shutil.copy(f"{run}/{f}", out)

In [ ]:
import json, glob
for c in sorted(glob.glob("runs/*/config.json")):
    print(c.split("/")[1], json.load(open(c)).get("val"))

In [ ]:
!cd /kaggle/working/runs && zip -qr /kaggle/working/real-encoders.zip .
!ls -la /kaggle/working/real-encoders.zip